In [1]:
import pandas as pd
from src.configuration.config import set_seed, SEED
from src.utils.data import au_cols
from src.training.evaluation import full_test_evaluation_per_split
from src.configuration.config import LOCKED_CONFIG
from src.training.training import full_training_per_split
from src.MILArchitecture.AttentionMIL import AttentionMIL
import torch
import torch.nn as nn
from src.training.loaders import initialize_loaders_per_split
from src.utils.data import sources
from src.MILArchitecture.windows import initialize_all_bags
from sklearn.metrics import balanced_accuracy_score, classification_report
from src.MILArchitecture.full_pipelines import attention_based_mil_combination_comparison

In [2]:
df = pd.read_csv("../data/processed/combined.csv")

# Testing ALL eight Single Signals

In [3]:
summary_single_signals = attention_based_mil_combination_comparison(df, au_cols)

In [4]:
summary_single_signals = summary_single_signals.sort_values("bal. accuracy", ascending=False)
summary_single_signals

,au,loss,bal. accuracy,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1
1,AU02_r,0.678007,0.559954,0.488889,0.640449,0.407407,0.7125,0.444444,0.674556
6,AU09_r,0.678225,0.548611,0.545455,0.625000,0.222222,0.8750,0.315789,0.729167
3,AU05_r,0.671638,0.492593,0.384615,0.592593,0.185185,0.8000,0.250000,0.680851
5,AU07_r,0.719458,0.486806,0.352941,0.589744,0.111111,0.8625,0.169014,0.700508
7,AU12_r,0.710709,0.445833,0.290323,0.563107,0.166667,0.7250,0.211765,0.633880
0,AU01_r,0.695597,0.436574,0.266667,0.557692,0.148148,0.7250,0.190476,0.630435
4,AU06_r,0.705670,0.417361,0.279070,0.538462,0.222222,0.6125,0.247423,0.573099
2,AU04_r,0.748692,0.392593,0.238095,0.521739,0.185185,0.6000,0.208333,0.558140


# Testing ALL AU Pairs

In [5]:
from itertools import combinations
au_pairs = list(combinations(au_cols, 2))

summary_pair_combinations = attention_based_mil_combination_comparison(df, au_pairs)

In [6]:
summary_pair_combinations

,au,loss,bal. accuracy,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1
12,"(AU02_r, AU12_r)",0.704791,0.618981,0.581395,0.681319,0.462963,0.7750,0.515464,0.725146
8,"(AU02_r, AU05_r)",0.713285,0.609259,0.538462,0.682927,0.518519,0.7000,0.528302,0.691358
23,"(AU06_r, AU09_r)",0.679796,0.606944,0.600000,0.666667,0.388889,0.8250,0.471910,0.737430
0,"(AU01_r, AU02_r)",0.699102,0.591435,0.571429,0.656566,0.370370,0.8125,0.449438,0.726257
27,"(AU09_r, AU12_r)",0.695724,0.582407,0.586207,0.647619,0.314815,0.8500,0.409639,0.735135
9,"(AU02_r, AU06_r)",0.723895,0.569907,0.548387,0.640777,0.314815,0.8250,0.400000,0.721311
13,"(AU04_r, AU05_r)",0.658919,0.566667,0.529412,0.640000,0.333333,0.8000,0.409091,0.711111
11,"(AU02_r, AU09_r)",0.704765,0.563194,0.500000,0.641304,0.388889,0.7375,0.437500,0.686047
18,"(AU05_r, AU06_r)",0.697032,0.560417,0.514286,0.636364,0.333333,0.7875,0.404494,0.703911
6,"(AU01_r, AU12_r)",0.686466,0.547222,0.461538,0.634146,0.444444,0.6500,0.452830,0.641975


# Testing ALL AU-Triplets

In [7]:
au_triples = list(combinations(au_cols, 3))

summary_triple_combinations = attention_based_mil_combination_comparison(df, au_triples)

In [8]:
summary_triple_combinations

,au,loss,bal. accuracy,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1
0,"(AU01_r, AU02_r, AU04_r)",0.757665,0.572917,0.545455,0.643564,0.333333,0.8125,0.413793,0.718232
31,"(AU02_r, AU06_r, AU09_r)",0.698890,0.571759,0.482759,0.657895,0.518519,0.6250,0.500000,0.641026
29,"(AU02_r, AU05_r, AU12_r)",0.695075,0.566204,0.500000,0.644444,0.407407,0.7250,0.448980,0.682353
35,"(AU02_r, AU09_r, AU12_r)",0.679280,0.560648,0.533333,0.634615,0.296296,0.8250,0.380952,0.717391
54,"(AU06_r, AU09_r, AU12_r)",0.724060,0.560417,0.514286,0.636364,0.333333,0.7875,0.404494,0.703911
52,"(AU06_r, AU07_r, AU09_r)",0.681624,0.552083,0.642857,0.625000,0.166667,0.9375,0.264706,0.750000
51,"(AU05_r, AU09_r, AU12_r)",0.688203,0.550926,0.487179,0.631579,0.351852,0.7500,0.408602,0.685714
1,"(AU01_r, AU02_r, AU05_r)",0.720552,0.550463,0.469388,0.635294,0.425926,0.6750,0.446602,0.654545
39,"(AU04_r, AU05_r, AU12_r)",0.678875,0.547917,0.486486,0.628866,0.333333,0.7625,0.395604,0.689266
21,"(AU02_r, AU04_r, AU05_r)",0.688427,0.544907,0.485714,0.626263,0.314815,0.7750,0.382022,0.692737


# Convert To LaTeX

In [9]:
summary_single_signals.to_latex(
    "../results/attention_based_mil/tables/single_signal_results.tex",
    index=True,
    float_format="%.3f"
)

summary_pair_combinations.to_latex(
    "../results/attention_based_mil/tables/pair_combination_results.tex",
    index=True,
    float_format="%.3f"
)

summary_triple_combinations.to_latex(
    "../results/attention_based_mil/tables/triple_combination_results.tex",
    index=True,
    float_format="%.3f"
)